# 04 · Registro full-cohort, biomarcadores y radiomics 3D

Procesa **todos los exámenes etiquetados** mediante un flujo streaming y reanudable:
cargar → orientar LPS → registrar → extraer features → actualizar mapas → liberar memoria.
No se retienen los 1.362 volúmenes en RAM.

Se prioriza un atlas validado si se configura `VALIDATED_MASK_PATH`. Sin atlas, se construye
una máscara de consenso a partir de normales representativos; sus columnas se mantienen como
`proxy` y no deben presentarse como biomarcadores clínicamente validados.

## 1. Configuración full-cohort

Los artefactos nuevos viven en `outputs/private_eda/full_cohort/`, por lo que no se mezclan
con los resultados del piloto anterior. Los checkpoints incluyen features, QC y acumuladores
online de media/varianza por clase.

In [ ]:
from __future__ import annotations

import atexit
import hashlib
import json
import math
import tempfile
import time
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import SimpleITK as sitk
import torch
import torch.nn.functional as torch_functional
from IPython.display import Markdown, display
from scipy import ndimage
from skimage.measure import marching_cubes, mesh_surface_area
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

NIFTI_ARCHIVE = PROJECT_ROOT / 'data' / 'raw' / 'niftis.zip'
PRIVATE_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'private_eda'
RUN_PROFILE = 'v3'
FULL_OUTPUT_DIR = PRIVATE_OUTPUT_DIR / f'full_cohort_{RUN_PROFILE}'
MANIFEST_PATH = PRIVATE_OUTPUT_DIR / 'image_qc_manifest.csv'

VALIDATED_MASK_PATH: Path | None = None
ATLAS_LABELS = {
    'right_target': 1,
    'left_target': 2,
    'background': 3,
    # Opcionales: right_caudate, left_caudate, right_putamen, left_putamen.
}
MAX_SCANS: int | None = None
REFERENCE_UID: str | None = None
REGISTRATION_MODE = 'rigid'
REGISTRATION_BACKEND = 'torch_cuda'
ISOTROPIC_SPACING_MM = 2.5
TORCH_REGISTRATION_STAGES = (
    (0.25, 28, 0.040),
    (0.50, 20, 0.025),
    (1.00, 12, 0.012),
)
TORCH_EARLY_STOPPING_PATIENCE = 7
GPU_FALLBACK_TO_SITK = True
SITK_METRIC_SAMPLING = 0.10
SITK_ITERATIONS = 100
MASK_TEMPLATE_SCANS = 48
TARGET_BASE_PERCENTILE = 90.0
TARGET_PERCENTILES = (88.0, 90.0, 92.0, 94.0)
ACTIVE_BACKGROUND_SD = 2.0
BACKGROUND_SCALE_FLOOR_FRACTION = 0.05
MIN_BACKGROUND_VOXELS = 64
MIN_BACKGROUND_SUPPORT_FRACTION = 0.20
TEXTURE_LEVELS = 32
CHECKPOINT_EVERY = 20
RESUME = True
RETRY_FAILURES = True
SAVE_REGISTERED_CROPS = True
CROP_MARGIN_MM = 20.0
RANDOM_SEED = 20260821

if REGISTRATION_BACKEND == 'torch_cuda' and not torch.cuda.is_available():
    display(Markdown(
        '**CUDA no disponible:** se utilizarÃ¡ SimpleITK CPU como fallback.'
    ))
    RESOLVED_REGISTRATION_BACKEND = 'simpleitk_cpu'
else:
    RESOLVED_REGISTRATION_BACKEND = REGISTRATION_BACKEND
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.benchmark = True

FEATURES_PATH = FULL_OUTPUT_DIR / 'dat_radiomics_features_full.csv'
REGISTRATION_PATH = FULL_OUTPUT_DIR / 'registration_qc_full.csv'
FAILURES_PATH = FULL_OUTPUT_DIR / 'registration_failures_full.csv'
STATE_PATH = FULL_OUTPUT_DIR / 'streaming_state_full.npz'
MAPS_PATH = FULL_OUTPUT_DIR / 'cohort_maps_full.npz'
MASKS_PATH = FULL_OUTPUT_DIR / 'analysis_masks_full.npz'
CONFIG_PATH = FULL_OUTPUT_DIR / 'registration_radiomics_full_config.json'
CROPS_DIR = FULL_OUTPUT_DIR / 'registered_crops'

if REGISTRATION_MODE not in {'rigid', 'affine'}:
    raise ValueError("REGISTRATION_MODE debe ser 'rigid' o 'affine'.")
if REGISTRATION_BACKEND not in {'torch_cuda', 'simpleitk_cpu'}:
    raise ValueError(
        "REGISTRATION_BACKEND debe ser 'torch_cuda' o 'simpleitk_cpu'."
    )
for required_path in [NIFTI_ARCHIVE, MANIFEST_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

FULL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if SAVE_REGISTERED_CROPS:
    CROPS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

## 2. Cohorte y referencia congelable

No se descarta el 5% técnicamente extremo. Todos los casos etiquetados se intentan procesar
y cualquier problema queda como bandera QC. Si `REFERENCE_UID=None`, la referencia se elige
determinísticamente entre normales de la familia de adquisición más frecuente; para una
evaluación definitiva conviene fijar explícitamente ese UID o usar un template externo.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH, dtype={'uid': 'string'})
required_columns = {
    'uid', 'member', 'is_pathologic', 'acquisition_family',
    'technical_outlier_score', 'within_protocol_qc_score',
    'shape_x', 'shape_y', 'shape_z',
    'spacing_x_mm', 'spacing_y_mm', 'spacing_z_mm',
    'fov_x_mm', 'fov_y_mm', 'fov_z_mm',
}
missing_columns = sorted(required_columns - set(manifest.columns))
if missing_columns:
    raise ValueError(
        'Ejecuta nuevamente el notebook 03 full-cohort. Faltan: '
        + ', '.join(missing_columns)
    )

cohort = manifest.loc[manifest['is_pathologic'].notna()].copy()
cohort['is_pathologic'] = cohort['is_pathologic'].astype(int)
cohort = cohort.sort_values('uid').reset_index(drop=True)
if MAX_SCANS is not None and MAX_SCANS < len(cohort):
    rng = np.random.default_rng(RANDOM_SEED)
    selected: list[int] = []
    per_class = max(1, MAX_SCANS // cohort['is_pathologic'].nunique())
    for _, group in cohort.groupby('is_pathologic'):
        selected.extend(
            rng.choice(group.index, size=min(per_class, len(group)), replace=False).tolist()
        )
    remaining = cohort.index.difference(selected)
    if len(selected) < MAX_SCANS:
        selected.extend(
            rng.choice(
                remaining,
                size=min(MAX_SCANS - len(selected), len(remaining)),
                replace=False,
            ).tolist()
        )
    cohort = cohort.loc[sorted(set(selected))].reset_index(drop=True)

member_by_uid = dict(zip(manifest['uid'].astype(str), manifest['member'].astype(str)))
largest_family = str(cohort['acquisition_family'].value_counts().idxmax())
if REFERENCE_UID is None:
    reference_candidates = cohort.loc[
        (cohort['is_pathologic'] == 0)
        & (cohort['acquisition_family'].astype(str) == largest_family)
    ].copy()
    if reference_candidates.empty:
        reference_candidates = cohort.loc[cohort['is_pathologic'] == 0].copy()
    if reference_candidates.empty:
        reference_candidates = cohort.copy()
    geometry_columns = [
        'spacing_x_mm', 'spacing_y_mm', 'spacing_z_mm',
        'fov_x_mm', 'fov_y_mm', 'fov_z_mm',
    ]
    geometry = reference_candidates[geometry_columns].astype(float)
    scale = geometry.std(ddof=0).replace(0, 1)
    reference_candidates['geometry_distance'] = np.sqrt(
        ((((geometry - geometry.median()) / scale) ** 2).mean(axis=1))
    )
    reference_candidates['reference_score'] = (
        reference_candidates['geometry_distance']
        + 0.20 * reference_candidates['within_protocol_qc_score'].fillna(0)
    )
    REFERENCE_UID = str(
        reference_candidates.nsmallest(1, 'reference_score').iloc[0]['uid']
    )
if REFERENCE_UID not in member_by_uid:
    raise ValueError(f'REFERENCE_UID={REFERENCE_UID!r} no existe en el manifiesto.')

display(Markdown(
    f'**Cohorte:** `{len(cohort):,}` · **referencia:** `{REFERENCE_UID}` · '
    f'**familia principal:** `{largest_family}`'
))
display(cohort.groupby('is_pathologic').size().rename('n').to_frame())

## 3. Registro físico LPS e isotropía

SimpleITK conserva origen, dirección y spacing y realiza la alineación geométrica inicial.
La referencia se remuestrea a 2,5 mm isotrópicos; el ajuste rígido residual se optimiza en
CUDA por correlación normalizada. Intensidades usan interpolación lineal; máscaras, vecino
más cercano. Los casos no fiables tienen fallback automático a SimpleITK/CPU.

In [ ]:
previous_archive = globals().get('_REGISTRATION_ARCHIVE')
if isinstance(previous_archive, zipfile.ZipFile):
    previous_archive.close()
previous_cache = globals().get('_REGISTRATION_CACHE')
if isinstance(previous_cache, tempfile.TemporaryDirectory):
    previous_cache.cleanup()

_REGISTRATION_CACHE = tempfile.TemporaryDirectory(
    prefix='dat_registration_cache_'
)
_REGISTRATION_CACHE_ROOT = Path(_REGISTRATION_CACHE.name)
_REGISTRATION_ARCHIVE = zipfile.ZipFile(NIFTI_ARCHIVE)


def cleanup_registration_cache() -> None:
    archive = globals().get('_REGISTRATION_ARCHIVE')
    if isinstance(archive, zipfile.ZipFile) and archive.fp is not None:
        archive.close()
    cache = globals().get('_REGISTRATION_CACHE')
    if isinstance(cache, tempfile.TemporaryDirectory):
        cache.cleanup()


atexit.register(cleanup_registration_cache)


def read_sitk_from_zip(
    member_name: str,
    pixel_type: int = sitk.sitkFloat32,
    keep_cached: bool = False,
) -> sitk.Image:
    extracted_path = _REGISTRATION_CACHE_ROOT / member_name
    if not extracted_path.exists():
        _REGISTRATION_ARCHIVE.extract(
            member_name, _REGISTRATION_CACHE_ROOT
        )
    try:
        image = sitk.ReadImage(str(extracted_path), pixel_type)
        oriented = sitk.Image(sitk.DICOMOrient(image, 'LPS'))
    finally:
        if not keep_cached:
            extracted_path.unlink(missing_ok=True)
    return oriented


def make_isotropic_reference(image: sitk.Image, spacing_mm: float) -> sitk.Image:
    old_size = np.asarray(image.GetSize(), dtype=float)
    old_spacing = np.asarray(image.GetSpacing(), dtype=float)
    new_spacing = np.repeat(float(spacing_mm), 3)
    new_size = np.maximum(1, np.rint(old_size * old_spacing / new_spacing)).astype(int)
    return sitk.Resample(
        image,
        [int(value) for value in new_size],
        sitk.Transform(3, sitk.sitkIdentity),
        sitk.sitkLinear,
        image.GetOrigin(),
        tuple(float(value) for value in new_spacing),
        image.GetDirection(),
        0.0,
        sitk.sitkFloat32,
    )


def rescale_for_registration(image: sitk.Image) -> sitk.Image:
    return sitk.RescaleIntensity(sitk.Cast(image, sitk.sitkFloat32), 0.0, 1.0)


def normalized_correlation(first: np.ndarray, second: np.ndarray) -> float:
    mask = np.isfinite(first) & np.isfinite(second) & ((first > 0) | (second > 0))
    if mask.sum() < 10:
        return float('nan')
    a, b = first[mask].astype(float), second[mask].astype(float)
    if a.std() <= 1e-8 or b.std() <= 1e-8:
        return float('nan')
    return float(np.corrcoef(a, b)[0, 1])


def configured_registration(
    fixed: sitk.Image, moving: sitk.Image, initial: sitk.Transform
) -> tuple[sitk.Transform, float]:
    method = sitk.ImageRegistrationMethod()
    method.SetMetricAsCorrelation()
    method.SetMetricSamplingStrategy(method.RANDOM)
    method.SetMetricSamplingPercentage(SITK_METRIC_SAMPLING, RANDOM_SEED)
    method.SetInterpolator(sitk.sitkLinear)
    method.SetOptimizerAsRegularStepGradientDescent(
        learningRate=1.0,
        minStep=1e-4,
        numberOfIterations=SITK_ITERATIONS,
        gradientMagnitudeTolerance=1e-8,
    )
    method.SetOptimizerScalesFromPhysicalShift()
    method.SetShrinkFactorsPerLevel(shrinkFactors=[4, 2, 1])
    method.SetSmoothingSigmasPerLevel(smoothingSigmas=[2, 1, 0])
    method.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
    method.SetInitialTransform(initial, inPlace=False)
    transform = method.Execute(
        rescale_for_registration(fixed), rescale_for_registration(moving)
    )
    return transform, float(method.GetMetricValue())


def register_to_reference_sitk(
    fixed: sitk.Image, moving: sitk.Image, mode: str
) -> tuple[sitk.Image, sitk.Transform, dict[str, object]]:
    started_at = time.perf_counter()
    rigid_initial = sitk.CenteredTransformInitializer(
        fixed,
        moving,
        sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY,
    )
    rigid, rigid_metric = configured_registration(fixed, moving, rigid_initial)
    final_transform: sitk.Transform = rigid
    final_metric = rigid_metric
    if mode == 'affine':
        affine_initial = sitk.AffineTransform(3)
        affine_initial.SetCenter(
            tuple((np.asarray(fixed.GetSize()) - 1) * np.asarray(fixed.GetSpacing()) / 2)
        )
        final_transform, final_metric = configured_registration(
            fixed, moving, sitk.CompositeTransform([rigid, affine_initial])
        )

    before = sitk.Resample(
        moving, fixed, sitk.Transform(3, sitk.sitkIdentity),
        sitk.sitkLinear, 0.0, sitk.sitkFloat32,
    )
    after = sitk.Resample(
        moving, fixed, final_transform, sitk.sitkLinear, 0.0, sitk.sitkFloat32
    )
    fixed_array = sitk.GetArrayViewFromImage(fixed)
    before_array = sitk.GetArrayViewFromImage(before)
    after_array = sitk.GetArrayViewFromImage(after)
    return after, final_transform, {
        'metric_final_correlation_objective': final_metric,
        'correlation_before': normalized_correlation(fixed_array, before_array),
        'correlation_after': normalized_correlation(fixed_array, after_array),
        'registration_backend': 'simpleitk_cpu',
        'gpu_fallback_used': False,
        'registration_seconds': time.perf_counter() - started_at,
    }


def torch_rotation_matrix(angles: torch.Tensor) -> torch.Tensor:
    ax, ay, az = angles
    one = torch.ones((), device=angles.device)
    zero = torch.zeros((), device=angles.device)
    cx, sx = torch.cos(ax), torch.sin(ax)
    cy, sy = torch.cos(ay), torch.sin(ay)
    cz, sz = torch.cos(az), torch.sin(az)
    rotation_x = torch.stack([
        one, zero, zero,
        zero, cx, -sx,
        zero, sx, cx,
    ]).reshape(3, 3)
    rotation_y = torch.stack([
        cy, zero, sy,
        zero, one, zero,
        -sy, zero, cy,
    ]).reshape(3, 3)
    rotation_z = torch.stack([
        cz, -sz, zero,
        sz, cz, zero,
        zero, zero, one,
    ]).reshape(3, 3)
    return rotation_z @ rotation_y @ rotation_x


def torch_warp(
    volume: torch.Tensor, parameters: torch.Tensor
) -> torch.Tensor:
    theta = torch.cat([
        torch_rotation_matrix(parameters[:3]),
        parameters[3:, None],
    ], dim=1)[None]
    grid = torch_functional.affine_grid(
        theta, volume.shape, align_corners=False
    )
    return torch_functional.grid_sample(
        volume, grid, mode='bilinear', padding_mode='zeros',
        align_corners=False,
    )


def torch_ncc_loss(
    fixed: torch.Tensor, moving: torch.Tensor
) -> torch.Tensor:
    valid = (fixed > 0.01) | (moving.detach() > 0.01)
    first = fixed[valid]
    second = moving[valid]
    if first.numel() < 10:
        return moving.sum() * 0 + 1
    first = first - first.mean()
    second = second - second.mean()
    denominator = torch.sqrt(
        first.square().mean() * second.square().mean()
    ).clamp_min(1e-6)
    return -(first * second).mean() / denominator


def register_to_reference_torch(
    fixed: sitk.Image, moving: sitk.Image
) -> tuple[sitk.Image, sitk.Transform, dict[str, object]]:
    started_at = time.perf_counter()
    initial = sitk.CenteredTransformInitializer(
        fixed,
        moving,
        sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY,
    )
    centered = sitk.Resample(
        moving, fixed, initial, sitk.sitkLinear, 0.0, sitk.sitkFloat32
    )
    fixed_array = sitk.GetArrayFromImage(
        rescale_for_registration(fixed)
    ).astype(np.float32)
    moving_original_array = sitk.GetArrayFromImage(centered).astype(np.float32)
    moving_array = sitk.GetArrayFromImage(
        rescale_for_registration(centered)
    ).astype(np.float32)
    correlation_before = normalized_correlation(
        fixed_array, moving_array
    )

    device = torch.device('cuda')
    fixed_tensor = torch.from_numpy(fixed_array)[None, None].to(device)
    moving_tensor = torch.from_numpy(moving_array)[None, None].to(device)
    parameters = torch.zeros(6, device=device, requires_grad=True)

    for scale, iterations, learning_rate in TORCH_REGISTRATION_STAGES:
        level_size = [
            max(12, int(round(size * scale)))
            for size in fixed_tensor.shape[2:]
        ]
        fixed_level = torch_functional.interpolate(
            fixed_tensor, size=level_size, mode='trilinear',
            align_corners=False,
        )
        moving_level = torch_functional.interpolate(
            moving_tensor, size=level_size, mode='trilinear',
            align_corners=False,
        )
        optimizer = torch.optim.Adam([parameters], lr=learning_rate)
        best_loss = float('inf')
        best_parameters = parameters.detach().clone()
        stale_iterations = 0
        for _ in range(iterations):
            optimizer.zero_grad(set_to_none=True)
            warped = torch_warp(moving_level, parameters)
            loss = torch_ncc_loss(fixed_level, warped)
            if not torch.isfinite(loss):
                raise RuntimeError('PÃ©rdida CUDA no finita.')
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                parameters[:3].clamp_(-0.35, 0.35)
                parameters[3:].clamp_(-0.35, 0.35)
            current_loss = float(loss.detach().cpu())
            if current_loss < best_loss - 1e-5:
                best_loss = current_loss
                best_parameters = parameters.detach().clone()
                stale_iterations = 0
            else:
                stale_iterations += 1
            if stale_iterations >= TORCH_EARLY_STOPPING_PATIENCE:
                break
        with torch.no_grad():
            parameters.copy_(best_parameters)

    with torch.no_grad():
        normalized_registered_array = (
            torch_warp(moving_tensor, parameters)
            .squeeze().cpu().numpy().astype(np.float32)
        )
        original_tensor = torch.from_numpy(
            moving_original_array
        )[None, None].to(device)
        registered_array = (
            torch_warp(original_tensor, parameters)
            .squeeze().cpu().numpy().astype(np.float32)
        )
    registered = sitk.GetImageFromArray(registered_array)
    registered.CopyInformation(fixed)
    correlation_after = normalized_correlation(
        fixed_array, normalized_registered_array
    )
    return registered, initial, {
        'metric_final_correlation_objective': -best_loss,
        'correlation_before': correlation_before,
        'correlation_after': correlation_after,
        'registration_backend': 'torch_cuda',
        'gpu_fallback_used': False,
        'registration_seconds': time.perf_counter() - started_at,
    }


def register_to_reference(
    fixed: sitk.Image, moving: sitk.Image, mode: str
) -> tuple[sitk.Image, sitk.Transform, dict[str, object]]:
    if RESOLVED_REGISTRATION_BACKEND != 'torch_cuda' or mode != 'rigid':
        return register_to_reference_sitk(fixed, moving, mode)
    try:
        registered, transform, metrics = register_to_reference_torch(
            fixed, moving
        )
        before = float(metrics['correlation_before'])
        after = float(metrics['correlation_after'])
        unreliable = (
            not np.isfinite(after)
            or after < 0.35
            or (np.isfinite(before) and after < before - 0.03)
        )
        if unreliable and GPU_FALLBACK_TO_SITK:
            fallback_image, fallback_transform, fallback_metrics = (
                register_to_reference_sitk(fixed, moving, mode)
            )
            fallback_metrics['gpu_fallback_used'] = True
            return fallback_image, fallback_transform, fallback_metrics
        return registered, transform, metrics
    except RuntimeError:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if not GPU_FALLBACK_TO_SITK:
            raise
        fallback_image, fallback_transform, fallback_metrics = (
            register_to_reference_sitk(fixed, moving, mode)
        )
        fallback_metrics['gpu_fallback_used'] = True
        return fallback_image, fallback_transform, fallback_metrics


native_reference = read_sitk_from_zip(member_by_uid[REFERENCE_UID])
reference_image = make_isotropic_reference(
    native_reference, ISOTROPIC_SPACING_MM
)
reference_array = sitk.GetArrayFromImage(reference_image).astype(np.float32)
display(Markdown(
    f'**Grid común (x,y,z):** `{reference_image.GetSize()}` · '
    f'**spacing:** `{reference_image.GetSpacing()}` mm · **orientación:** LPS · '
    f'**backend:** `{RESOLVED_REGISTRATION_BACKEND}`'
))

## 4. Máscaras: atlas validado o consenso de normales

El fallback registra hasta 48 normales representativos y promedia intensidades normalizadas
online. Conserva el componente dominante en cada hemisferio y define un fondo posterior de
baja captación. Sigue siendo una aproximación: `mask_mode` permite impedir que se confunda con
un atlas anatómico.

In [ ]:
def normalize_display(array: np.ndarray) -> np.ndarray:
    positive = array[np.isfinite(array) & (array > 0)]
    values = positive if positive.size else array[np.isfinite(array)]
    low, high = np.percentile(values, [1, 99.5])
    return np.clip((array - low) / max(high - low, 1e-8), 0, 1)


def representative_template_rows(frame: pd.DataFrame, total: int) -> pd.DataFrame:
    normals = frame.loc[frame['is_pathologic'] == 0].copy()
    pool = normals if len(normals) else frame.copy()
    pool = pool.sort_values(['within_protocol_qc_score', 'uid'])
    pieces = []
    family_count = max(1, pool['acquisition_family'].nunique())
    per_family = max(1, math.ceil(total / family_count))
    for _, group in pool.groupby('acquisition_family', sort=True):
        pieces.append(group.head(per_family))
    return (
        pd.concat(pieces, ignore_index=True)
        .sort_values(['within_protocol_qc_score', 'uid'])
        .head(total)
    )


template_rows = representative_template_rows(cohort, MASK_TEMPLATE_SCANS)
if REFERENCE_UID not in set(template_rows['uid'].astype(str)):
    template_rows = pd.concat([
        cohort.loc[cohort['uid'].astype(str) == REFERENCE_UID],
        template_rows.iloc[:-1] if len(template_rows) else template_rows,
    ]).drop_duplicates('uid')

consensus = np.zeros(reference_array.shape, dtype=np.float64)
consensus_count = 0
template_failures: list[dict[str, str]] = []
for row in tqdm(
    template_rows.itertuples(index=False),
    total=len(template_rows),
    desc='Construyendo consenso',
):
    uid = str(row.uid)
    try:
        if uid == REFERENCE_UID:
            registered_image = sitk.Image(reference_image)
        else:
            moving = read_sitk_from_zip(
                member_by_uid[uid], keep_cached=True
            )
            registered_image, _, _ = register_to_reference(
                reference_image, moving, REGISTRATION_MODE
            )
        candidate = normalize_display(
            sitk.GetArrayViewFromImage(registered_image)
        )
        consensus_count += 1
        consensus += (candidate - consensus) / consensus_count
    except Exception as error:
        template_failures.append({
            'uid': uid, 'error': f'{type(error).__name__}: {error}'
        })
if consensus_count < 2:
    raise RuntimeError('No se pudo construir un consenso con al menos dos exámenes.')
consensus = consensus.astype(np.float32)


def central_box(shape_zyx: tuple[int, int, int]) -> np.ndarray:
    z_size, y_size, x_size = shape_zyx
    mask = np.zeros(shape_zyx, dtype=bool)
    mask[
        int(0.20 * z_size):int(0.80 * z_size),
        int(0.20 * y_size):int(0.80 * y_size),
        int(0.20 * x_size):int(0.80 * x_size),
    ] = True
    return mask


def central_ellipsoid(shape_zyx: tuple[int, int, int]) -> np.ndarray:
    coordinates = np.indices(shape_zyx, dtype=np.float32)
    center = (np.asarray(shape_zyx, dtype=np.float32) - 1) / 2
    radii = np.maximum(np.asarray(shape_zyx, dtype=np.float32) * 0.42, 1)
    distance = sum(
        ((coordinates[axis] - center[axis]) / radii[axis]) ** 2
        for axis in range(3)
    )
    return distance <= 1


def largest_component(mask: np.ndarray) -> np.ndarray:
    labels, count = ndimage.label(mask)
    if count == 0:
        return np.zeros_like(mask, dtype=bool)
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    return labels == int(np.argmax(sizes))


def bilateral_target(array: np.ndarray, percentile: float) -> np.ndarray:
    positive = np.isfinite(array) & (array > 0)
    candidates = central_box(array.shape) & positive
    values = array[candidates]
    if values.size < 20:
        raise ValueError('Consenso insuficiente para máscara objetivo.')
    raw = candidates & (array >= np.percentile(values, percentile))
    x_mid = array.shape[2] // 2
    target = np.zeros_like(raw)
    target[:, :, :x_mid] = largest_component(raw[:, :, :x_mid])
    target[:, :, x_mid:] = largest_component(raw[:, :, x_mid:])
    target = ndimage.binary_closing(target, iterations=1)
    return ndimage.binary_fill_holes(target)


def proxy_masks_from_consensus(
    array: np.ndarray, percentile: float
) -> dict[str, np.ndarray]:
    target = bilateral_target(array, percentile)
    positive = np.isfinite(array) & (array > 0)
    support_values = array[positive]
    support = positive & (array >= np.percentile(support_values, 10))
    support = ndimage.binary_closing(support, iterations=2)
    brain = largest_component(support & central_ellipsoid(array.shape))
    brain = ndimage.binary_fill_holes(
        ndimage.binary_closing(brain, iterations=2)
    )
    if brain.sum() < max(100, target.sum() * 2):
        brain = support & central_box(array.shape)
    posterior_band = np.zeros_like(target)
    posterior_band[
        int(0.15 * target.shape[0]):int(0.85 * target.shape[0]),
        int(0.55 * target.shape[1]):int(0.95 * target.shape[1]),
        int(0.10 * target.shape[2]):int(0.90 * target.shape[2]),
    ] = True
    exclusion = ndimage.binary_dilation(target, iterations=3)
    brain_values = array[brain & positive]
    low_cut = np.percentile(brain_values, 70)
    background = posterior_band & brain & (~exclusion) & (array <= low_cut)
    if background.sum() < max(20, target.sum() // 2):
        background = brain & (~exclusion) & (array <= low_cut)

    x_mid = target.shape[2] // 2
    y_mid = target.shape[1] // 2
    right = target.copy(); right[:, :, x_mid:] = False
    left = target.copy(); left[:, :, :x_mid] = False
    anterior = target.copy(); anterior[:, y_mid:, :] = False
    posterior = target.copy(); posterior[:, :y_mid, :] = False
    if min(target.sum(), background.sum(), right.sum(), left.sum()) == 0:
        raise ValueError('La máscara de consenso quedó vacía en una región requerida.')
    return {
        'target': target, 'background': background, 'brain': brain,
        'right': right, 'left': left,
        'anterior': anterior, 'posterior': posterior,
    }


def load_validated_masks(path: Path) -> dict[str, np.ndarray]:
    atlas = sitk.DICOMOrient(sitk.ReadImage(str(path), sitk.sitkUInt16), 'LPS')
    atlas = sitk.Resample(
        atlas, reference_image, sitk.Transform(3, sitk.sitkIdentity),
        sitk.sitkNearestNeighbor, 0,
    )
    labels = sitk.GetArrayFromImage(atlas)
    right = labels == ATLAS_LABELS['right_target']
    left = labels == ATLAS_LABELS['left_target']
    background = labels == ATLAS_LABELS['background']
    target = right | left
    brain = labels > 0
    y_mid = labels.shape[1] // 2
    result = {
        'target': target, 'background': background, 'brain': brain,
        'right': right, 'left': left,
        'anterior': target & (np.indices(target.shape)[1] < y_mid),
        'posterior': target & (np.indices(target.shape)[1] >= y_mid),
    }
    optional_labels = [
        'right_caudate', 'left_caudate', 'right_putamen', 'left_putamen'
    ]
    for name in optional_labels:
        if name in ATLAS_LABELS:
            result[name] = labels == ATLAS_LABELS[name]
    if min(target.sum(), background.sum(), right.sum(), left.sum()) == 0:
        raise ValueError('El atlas no contiene todas las regiones requeridas.')
    return result


if VALIDATED_MASK_PATH is not None:
    if not VALIDATED_MASK_PATH.exists():
        raise FileNotFoundError(VALIDATED_MASK_PATH)
    masks = load_validated_masks(VALIDATED_MASK_PATH)
    sensitivity_targets = {'atlas_base': masks['target']}
    mask_mode = 'validated_atlas'
else:
    masks = proxy_masks_from_consensus(consensus, TARGET_BASE_PERCENTILE)
    sensitivity_targets = {
        f'p{int(percentile)}': proxy_masks_from_consensus(consensus, percentile)['target']
        for percentile in TARGET_PERCENTILES
    }
    mask_mode = 'consensus_uptake_proxy'

with MASKS_PATH.open('wb') as stream:
    np.savez_compressed(
        stream,
        consensus=consensus,
        **{name: value.astype(np.uint8) for name, value in masks.items()},
    )
display(Markdown(
    f'**Modo de máscara:** `{mask_mode}` · **template:** `{consensus_count}` · '
    f'**target:** `{masks["target"].sum():,}` voxels · '
    f'**background:** `{masks["background"].sum():,}` voxels'
))

## 5. Features semicuantitativas, forma, textura 3D y sensibilidad

El fondo se intersecta con el soporte positivo de cada estudio y dispone de fallbacks y un
piso relativo al percentil 90 del foreground. Se conservan tanto la razón robusta como flags
de validez; ningún valor se presenta como SBR clínicamente validado. La textura usa una GLCM
3D de 32 niveles y 13 direcciones y todavía requiere verificación IBSI.

In [ ]:
def masked_values(array: np.ndarray, mask: np.ndarray) -> np.ndarray:
    return array[mask & np.isfinite(array)]


def safe_mean(array: np.ndarray, mask: np.ndarray) -> float:
    values = masked_values(array, mask)
    return float(values.mean()) if values.size else float('nan')


def sbr_like(target_mean: float, background_mean: float) -> float:
    if not np.isfinite(background_mean) or background_mean <= 1e-8:
        return float('nan')
    return float((target_mean - background_mean) / background_mean)


def robust_background_context(
    array: np.ndarray, base_masks: dict[str, np.ndarray]
) -> dict[str, object]:
    positive = np.isfinite(array) & (array > 0)
    brain_mask = base_masks.get('brain', central_ellipsoid(array.shape))
    foreground_mask = brain_mask & positive
    if foreground_mask.sum() < MIN_BACKGROUND_VOXELS:
        foreground_mask = central_ellipsoid(array.shape) & positive
    if foreground_mask.sum() < MIN_BACKGROUND_VOXELS:
        foreground_mask = positive
    foreground_values = array[foreground_mask]
    if foreground_values.size < MIN_BACKGROUND_VOXELS:
        raise ValueError('Foreground positivo insuficiente después del registro.')

    foreground_p50, foreground_p90, foreground_p99 = np.percentile(
        foreground_values, [50, 90, 99]
    )
    scale_floor = max(
        1e-8,
        BACKGROUND_SCALE_FLOOR_FRACTION * float(foreground_p90),
    )
    nominal_background = base_masks['background']
    candidate = nominal_background & positive
    support_fraction = float(candidate.sum() / max(nominal_background.sum(), 1))
    source = 'consensus_brain_background'
    enough_support = (
        candidate.sum() >= MIN_BACKGROUND_VOXELS
        and support_fraction >= MIN_BACKGROUND_SUPPORT_FRACTION
    )
    if not enough_support:
        exclusion = ndimage.binary_dilation(
            base_masks['target'], iterations=5
        )
        candidate = foreground_mask & (~exclusion)
        if candidate.any():
            candidate &= array <= np.percentile(array[candidate], 70)
        source = 'scan_brain_fallback'

    values = array[candidate & positive]
    if values.size < MIN_BACKGROUND_VOXELS:
        values = foreground_values[
            foreground_values <= np.percentile(foreground_values, 50)
        ]
        source = 'foreground_lower_half_fallback'
    if values.size < 10:
        raise ValueError('Fondo robusto insuficiente después del registro.')

    low, high = np.percentile(values, [5, 95])
    trimmed = values[(values >= low) & (values <= high)]
    if trimmed.size < 10:
        trimmed = values
    raw_mean = float(trimmed.mean())
    raw_std = float(trimmed.std(ddof=1)) if trimmed.size > 1 else 0.0
    floor_applied = (not np.isfinite(raw_mean)) or raw_mean < scale_floor
    scale = max(raw_mean if np.isfinite(raw_mean) else 0.0, scale_floor)
    valid = bool(
        enough_support
        and source == 'consensus_brain_background'
        and not floor_applied
        and np.isfinite(raw_std)
    )
    return {
        'values': trimmed,
        'source': source,
        'support_voxels': int(values.size),
        'support_fraction': support_fraction,
        'mean_raw': raw_mean,
        'std_raw': raw_std,
        'scale_floor': float(scale_floor),
        'scale': float(scale),
        'floor_applied': bool(floor_applied),
        'qc_valid': valid,
        'foreground_p50': float(foreground_p50),
        'foreground_p90': float(foreground_p90),
        'foreground_p99': float(foreground_p99),
    }


def largest_component_shape(
    mask: np.ndarray, spacing_xyz: tuple[float, float, float]
) -> dict[str, float]:
    labels, components = ndimage.label(mask)
    if components == 0:
        return {
            'volume_ml': 0.0, 'surface_mm2': float('nan'),
            'elongation': float('nan'), 'sphericity': float('nan'),
            'extent': float('nan'), 'components': 0,
            'largest_component_fraction': 0.0,
        }
    sizes = np.bincount(labels.ravel())[1:]
    largest = labels == (int(np.argmax(sizes)) + 1)
    coords = np.argwhere(largest)
    spacing_zyx = np.asarray(spacing_xyz[::-1], dtype=float)
    physical = coords * spacing_zyx
    eigenvalues = np.linalg.eigvalsh(np.cov(physical, rowvar=False))
    elongation = math.sqrt(
        max(float(eigenvalues[-1]), 0) / max(float(eigenvalues[0]), 1e-8)
    )
    volume_mm3 = float(largest.sum() * np.prod(spacing_xyz))
    bbox_size = (coords.max(axis=0) - coords.min(axis=0) + 1) * spacing_zyx
    bbox_volume = float(np.prod(bbox_size))
    try:
        vertices, faces, _, _ = marching_cubes(
            largest.astype(np.uint8), level=0.5, spacing=tuple(spacing_zyx)
        )
        surface = float(mesh_surface_area(vertices, faces))
    except (ValueError, RuntimeError):
        surface = float('nan')
    sphericity = (
        float((math.pi ** (1 / 3)) * ((6 * volume_mm3) ** (2 / 3)) / surface)
        if np.isfinite(surface) and surface > 0 else float('nan')
    )
    return {
        'volume_ml': volume_mm3 / 1000,
        'surface_mm2': surface,
        'elongation': float(elongation),
        'sphericity': sphericity,
        'extent': volume_mm3 / max(bbox_volume, 1e-8),
        'components': int(components),
        'largest_component_fraction': float(sizes.max() / max(mask.sum(), 1)),
    }


OFFSETS_3D = [
    (1, 0, 0), (0, 1, 0), (0, 0, 1),
    (1, 1, 0), (1, -1, 0), (1, 0, 1), (1, 0, -1),
    (0, 1, 1), (0, 1, -1),
    (1, 1, 1), (1, 1, -1), (1, -1, 1), (1, -1, -1),
]


def paired_slices(size: int, offset: int) -> tuple[slice, slice]:
    if offset > 0:
        return slice(0, size - offset), slice(offset, size)
    if offset < 0:
        return slice(-offset, size), slice(0, size + offset)
    return slice(0, size), slice(0, size)


def texture_glcm_3d(
    ratio: np.ndarray, mask: np.ndarray, levels: int = 32
) -> dict[str, float]:
    values = masked_values(ratio, mask)
    names = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'entropy', 'correlation']
    if values.size < 30 or float(values.max()) <= float(values.min()):
        return {name: float('nan') for name in names}
    low, high = np.percentile(values, [1, 99])
    quantized = np.rint(
        np.clip((ratio - low) / max(high - low, 1e-8), 0, 1) * (levels - 1)
    ).astype(np.uint8)
    matrix = np.zeros((levels, levels), dtype=np.float64)
    for dz, dy, dx in OFFSETS_3D:
        z0, z1 = paired_slices(mask.shape[0], dz)
        y0, y1 = paired_slices(mask.shape[1], dy)
        x0, x1 = paired_slices(mask.shape[2], dx)
        valid = mask[z0, y0, x0] & mask[z1, y1, x1]
        if not valid.any():
            continue
        first = quantized[z0, y0, x0][valid]
        second = quantized[z1, y1, x1][valid]
        np.add.at(matrix, (first, second), 1)
        np.add.at(matrix, (second, first), 1)
    if matrix.sum() == 0:
        return {name: float('nan') for name in names}
    probability = matrix / matrix.sum()
    i, j = np.indices(probability.shape)
    contrast = float((probability * (i - j) ** 2).sum())
    dissimilarity = float((probability * np.abs(i - j)).sum())
    homogeneity = float((probability / (1 + (i - j) ** 2)).sum())
    energy = float(np.sqrt((probability ** 2).sum()))
    nonzero = probability[probability > 0]
    texture_entropy = float(-(nonzero * np.log2(nonzero)).sum())
    row = probability.sum(axis=1)
    column = probability.sum(axis=0)
    mean_i = float((np.arange(levels) * row).sum())
    mean_j = float((np.arange(levels) * column).sum())
    std_i = math.sqrt(float((((np.arange(levels) - mean_i) ** 2) * row).sum()))
    std_j = math.sqrt(float((((np.arange(levels) - mean_j) ** 2) * column).sum()))
    correlation = float(
        (probability * (i - mean_i) * (j - mean_j)).sum()
        / max(std_i * std_j, 1e-8)
    )
    return {
        'contrast': contrast, 'dissimilarity': dissimilarity,
        'homogeneity': homogeneity, 'energy': energy,
        'entropy': texture_entropy, 'correlation': correlation,
    }


def shifted_mask(mask: np.ndarray, shift_zyx: tuple[int, int, int]) -> np.ndarray:
    return ndimage.shift(
        mask.astype(np.uint8), shift=shift_zyx,
        order=0, mode='constant', cval=0,
    ).astype(bool)


def feature_record(
    uid: str, array: np.ndarray, base_masks: dict[str, np.ndarray]
) -> tuple[dict[str, object], np.ndarray, np.ndarray]:
    background = robust_background_context(array, base_masks)
    background_mean = float(background['scale'])
    background_std = float(background['std_raw'])
    ratio = array / background_mean
    intensity01 = np.clip(
        array / max(float(background['foreground_p99']), 1e-8), 0, 1
    )

    target_mean = safe_mean(array, base_masks['target'])
    right_mean = safe_mean(array, base_masks['right'])
    left_mean = safe_mean(array, base_masks['left'])
    anterior_mean = safe_mean(array, base_masks['anterior'])
    posterior_mean = safe_mean(array, base_masks['posterior'])
    side_mean = max((left_mean + right_mean) / 2, 1e-8)
    target_ratio = masked_values(ratio, base_masks['target'])

    active_threshold = background_mean + ACTIVE_BACKGROUND_SD * background_std
    active_mask = base_masks['target'] & np.isfinite(array) & (array >= active_threshold)
    active_mask = ndimage.binary_closing(active_mask, iterations=1)
    shape = largest_component_shape(active_mask, reference_image.GetSpacing())
    texture = texture_glcm_3d(ratio, base_masks['target'], TEXTURE_LEVELS)

    threshold_sbr = [
        sbr_like(safe_mean(array, candidate), background_mean)
        for candidate in sensitivity_targets.values()
    ]
    morphology_masks = [
        ndimage.binary_erosion(base_masks['target'], iterations=1),
        base_masks['target'],
        ndimage.binary_dilation(base_masks['target'], iterations=1),
    ]
    morphology_sbr = [
        sbr_like(safe_mean(array, candidate), background_mean)
        for candidate in morphology_masks
    ]
    translation_sbr = [
        sbr_like(safe_mean(array, shifted_mask(base_masks['target'], shift)), background_mean)
        for shift in [(0, 0, 0), (0, 0, 1), (0, 0, -1), (0, 1, 0), (0, -1, 0)]
    ]
    background_means = [
        safe_mean(array, candidate)
        for candidate in [
            ndimage.binary_erosion(base_masks['background'], iterations=1),
            base_masks['background'],
            ndimage.binary_dilation(base_masks['background'], iterations=1),
        ]
    ]
    background_sbr = [
        sbr_like(target_mean, max(value, float(background['scale_floor'])))
        if np.isfinite(value) else float('nan')
        for value in background_means
    ]

    record: dict[str, object] = {
        'uid': uid,
        'mask_mode': mask_mode,
        'background_source': str(background['source']),
        'background_qc_valid': bool(background['qc_valid']),
        'background_floor_applied': bool(background['floor_applied']),
        'background_support_voxels': int(background['support_voxels']),
        'background_support_fraction': float(background['support_fraction']),
        'background_mean_raw': float(background['mean_raw']),
        'background_std_raw': background_std,
        'background_scale_floor': float(background['scale_floor']),
        'background_scale_used': background_mean,
        'foreground_p50_registered': float(background['foreground_p50']),
        'foreground_p90_registered': float(background['foreground_p90']),
        'foreground_p99_registered': float(background['foreground_p99']),
        'semiquant_sbr': sbr_like(target_mean, background_mean),
        'semiquant_right_sbr': sbr_like(right_mean, background_mean),
        'semiquant_left_sbr': sbr_like(left_mean, background_mean),
        'semiquant_min_side_sbr': min(
            sbr_like(right_mean, background_mean),
            sbr_like(left_mean, background_mean),
        ),
        'semiquant_lr_asymmetry_abs': abs(left_mean - right_mean) / side_mean,
        'semiquant_lr_asymmetry_signed': (left_mean - right_mean) / side_mean,
        'semiquant_posterior_anterior_ratio': posterior_mean / max(anterior_mean, 1e-8),
        'semiquant_log_target_background': float(
            np.log1p(max(target_mean, 0) / background_mean)
        ),
        'firstorder_ratio_mean': float(target_ratio.mean()),
        'firstorder_ratio_std': float(target_ratio.std(ddof=1)),
        'firstorder_ratio_p10': float(np.percentile(target_ratio, 10)),
        'firstorder_ratio_p50': float(np.percentile(target_ratio, 50)),
        'firstorder_ratio_p90': float(np.percentile(target_ratio, 90)),
        'active_threshold_sbr': float(
            (active_threshold - background_mean) / background_mean
        ),
        **{f'shape_active_{name}': value for name, value in shape.items()},
        **{f'texture3d_{name}': value for name, value in texture.items()},
        'stability_threshold_span': float(np.nanmax(threshold_sbr) - np.nanmin(threshold_sbr)),
        'stability_mask_span': float(np.nanmax(morphology_sbr) - np.nanmin(morphology_sbr)),
        'stability_translation_span': float(np.nanmax(translation_sbr) - np.nanmin(translation_sbr)),
        'stability_background_span': float(np.nanmax(background_sbr) - np.nanmin(background_sbr)),
    }
    if all(name in base_masks for name in [
        'right_caudate', 'left_caudate', 'right_putamen', 'left_putamen'
    ]):
        caudate = safe_mean(
            array, base_masks['right_caudate'] | base_masks['left_caudate']
        )
        putamen = safe_mean(
            array, base_masks['right_putamen'] | base_masks['left_putamen']
        )
        record['semiquant_putamen_caudate_ratio'] = putamen / max(caudate, 1e-8)
    return record, ratio.astype(np.float32), intensity01.astype(np.float32)

## 6. Ejecución streaming, checkpoints y crops

Los crops guardan tanto la razón robusta contra fondo como una intensidad acotada por el
percentil 99 del foreground. Comparten el grid de referencia y se almacenan en `float16`.
Un fondo dudoso queda marcado, pero ya no elimina silenciosamente el estudio.

In [ ]:
config_payload = {
    'algorithm_version': 'full_cohort_v3',
    'run_profile': RUN_PROFILE,
    'reference_uid': REFERENCE_UID,
    'registration_mode': REGISTRATION_MODE,
    'registration_backend_requested': REGISTRATION_BACKEND,
    'registration_backend_resolved': RESOLVED_REGISTRATION_BACKEND,
    'torch_registration_stages': [list(stage) for stage in TORCH_REGISTRATION_STAGES],
    'torch_early_stopping_patience': TORCH_EARLY_STOPPING_PATIENCE,
    'gpu_fallback_to_sitk': GPU_FALLBACK_TO_SITK,
    'sitk_metric_sampling': SITK_METRIC_SAMPLING,
    'sitk_iterations': SITK_ITERATIONS,
    'checkpoint_every': CHECKPOINT_EVERY,
    'retry_failures': RETRY_FAILURES,
    'cuda_device': (
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    ),
    'isotropic_spacing_mm': ISOTROPIC_SPACING_MM,
    'mask_mode': mask_mode,
    'validated_mask_path': str(VALIDATED_MASK_PATH) if VALIDATED_MASK_PATH else None,
    'atlas_labels': ATLAS_LABELS if VALIDATED_MASK_PATH else None,
    'target_base_percentile': TARGET_BASE_PERCENTILE,
    'target_percentiles': list(TARGET_PERCENTILES),
    'active_background_sd': ACTIVE_BACKGROUND_SD,
    'background_scale_floor_fraction': BACKGROUND_SCALE_FLOOR_FRACTION,
    'min_background_voxels': MIN_BACKGROUND_VOXELS,
    'min_background_support_fraction': MIN_BACKGROUND_SUPPORT_FRACTION,
    'texture_levels': TEXTURE_LEVELS,
    'max_scans': MAX_SCANS,
    'cohort_uid_hash': hashlib.sha256(
        '|'.join(cohort['uid'].astype(str)).encode('utf-8')
    ).hexdigest(),
}
config_hash = hashlib.sha256(
    json.dumps(config_payload, sort_keys=True).encode('utf-8')
).hexdigest()
config_payload['config_hash'] = config_hash

if CONFIG_PATH.exists() and RESUME:
    previous_config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
    if previous_config.get('config_hash') != config_hash:
        raise RuntimeError(
            'Existe un checkpoint full-cohort con otra configuración. '
            f'Archiva {FULL_OUTPUT_DIR} o restaura sus parámetros antes de continuar.'
        )
else:
    CONFIG_PATH.write_text(
        json.dumps(config_payload, indent=2), encoding='utf-8'
    )

feature_rows = (
    pd.read_csv(FEATURES_PATH).to_dict('records')
    if RESUME and FEATURES_PATH.exists() else []
)
registration_rows = (
    pd.read_csv(REGISTRATION_PATH).to_dict('records')
    if RESUME and REGISTRATION_PATH.exists() else []
)
failure_rows = (
    pd.read_csv(FAILURES_PATH).to_dict('records')
    if RESUME and FAILURES_PATH.exists() else []
)

map_state = {
    0: {'n': 0, 'mean': np.zeros(reference_array.shape, dtype=np.float64),
        'm2': np.zeros(reference_array.shape, dtype=np.float64)},
    1: {'n': 0, 'mean': np.zeros(reference_array.shape, dtype=np.float64),
        'm2': np.zeros(reference_array.shape, dtype=np.float64)},
}
mapped_uids: set[str] = set()
legacy_saved = globals().get('saved')
if isinstance(legacy_saved, np.lib.npyio.NpzFile):
    legacy_saved.close()
if RESUME and STATE_PATH.exists():
    with np.load(STATE_PATH, allow_pickle=False) as saved_state:
        for label in [0, 1]:
            map_state[label]['n'] = int(saved_state[f'n{label}'])
            map_state[label]['mean'] = saved_state[f'mean{label}'].astype(np.float64)
            map_state[label]['m2'] = saved_state[f'm2_{label}'].astype(np.float64)
        if 'mapped_uids' in saved_state.files:
            mapped_uids = {
                str(uid) for uid in saved_state['mapped_uids'].astype(str)
            }

    if not mapped_uids and sum(map_state[label]['n'] for label in [0, 1]):
        feature_frame_resume = pd.DataFrame(feature_rows)
        for label in [0, 1]:
            available = set(
                feature_frame_resume.loc[
                    feature_frame_resume['is_pathologic'].astype(int) == label,
                    'uid',
                ].astype(str)
            )
            ordered = [
                str(uid)
                for uid in cohort.loc[
                    cohort['is_pathologic'] == label, 'uid'
                ].astype(str)
                if str(uid) in available
            ]
            expected = int(map_state[label]['n'])
            if expected > len(ordered):
                raise RuntimeError(
                    'El estado online contiene mÃ¡s casos que la tabla de features.'
                )
            mapped_uids.update(ordered[:expected])
elif feature_rows:
    raise RuntimeError('Hay features reanudables, pero falta streaming_state_full.npz.')

label_by_uid = dict(zip(
    cohort['uid'].astype(str), cohort['is_pathologic'].astype(int)
))
mapped_counts = {
    label: sum(label_by_uid.get(uid) == label for uid in mapped_uids)
    for label in [0, 1]
}
if any(mapped_counts[label] != map_state[label]['n'] for label in [0, 1]):
    raise RuntimeError(
        'Los UID del estado online no coinciden con sus conteos por clase.'
    )

def online_update(label: int, uid: str, array: np.ndarray) -> None:
    if uid in mapped_uids:
        return
    state = map_state[int(label)]
    state['n'] += 1
    delta = array - state['mean']
    state['mean'] += delta / state['n']
    state['m2'] += delta * (array - state['mean'])
    mapped_uids.add(uid)


crop_indices = np.argwhere(masks['target'])
spacing_zyx = np.asarray(reference_image.GetSpacing()[::-1])
margin = np.ceil(CROP_MARGIN_MM / spacing_zyx).astype(int)
crop_start = np.maximum(crop_indices.min(axis=0) - margin, 0)
crop_stop = np.minimum(crop_indices.max(axis=0) + margin + 1, masks['target'].shape)
crop_slices = tuple(
    slice(int(start), int(stop)) for start, stop in zip(crop_start, crop_stop)
)


def save_crop(
    uid: str, ratio: np.ndarray, intensity01: np.ndarray
) -> None:
    destination = CROPS_DIR / f'{uid}.npz'
    if destination.exists() and RESUME:
        try:
            with np.load(destination, allow_pickle=False) as saved_crop:
                if {'volume_ratio', 'volume_intensity01'} <= set(saved_crop.files):
                    return
        except (OSError, ValueError, EOFError, zipfile.BadZipFile):
            pass
    temporary = destination.with_suffix('.npz.tmp')
    with temporary.open('wb') as stream:
        np.savez_compressed(
            stream,
            volume_ratio=np.clip(ratio[crop_slices], 0, 20).astype(np.float16),
            volume_intensity01=intensity01[crop_slices].astype(np.float16),
            target_mask=masks['target'][crop_slices].astype(np.uint8),
            crop_start_zyx=crop_start.astype(np.int16),
            spacing_xyz=np.asarray(reference_image.GetSpacing(), dtype=np.float32),
        )
    replace_with_retry(temporary, destination)


def replace_with_retry(
    temporary: Path, destination: Path, attempts: int = 10
) -> None:
    delay_seconds = 0.25
    for attempt in range(attempts):
        try:
            temporary.replace(destination)
            return
        except PermissionError as error:
            if attempt == attempts - 1:
                raise PermissionError(
                    f'No se pudo reemplazar {destination}. Cierra visores/Excel que '
                    'tengan abierto el archivo y pausa temporalmente OneDrive.'
                ) from error
            time.sleep(delay_seconds)
            delay_seconds = min(delay_seconds * 2, 5.0)


def atomic_csv(rows: list[dict[str, object]], path: Path, columns=None) -> None:
    temporary = path.with_suffix(path.suffix + '.tmp')
    pd.DataFrame(rows, columns=columns).to_csv(temporary, index=False)
    replace_with_retry(temporary, path)


def checkpoint() -> None:
    features_frame = pd.DataFrame(feature_rows).drop_duplicates('uid', keep='last')
    registration_frame = pd.DataFrame(registration_rows).drop_duplicates('uid', keep='last')
    temporary_features = FEATURES_PATH.with_suffix('.csv.tmp')
    temporary_registration = REGISTRATION_PATH.with_suffix('.csv.tmp')
    features_frame.sort_values('uid').to_csv(temporary_features, index=False)
    registration_frame.sort_values('uid').to_csv(temporary_registration, index=False)
    replace_with_retry(temporary_features, FEATURES_PATH)
    replace_with_retry(temporary_registration, REGISTRATION_PATH)
    atomic_csv(failure_rows, FAILURES_PATH, columns=['uid', 'error'])
    temporary_state = STATE_PATH.with_suffix('.npz.tmp')
    uid_width = max((len(uid) for uid in mapped_uids), default=1)
    with temporary_state.open('wb') as stream:
        np.savez_compressed(
            stream,
            n0=np.asarray(map_state[0]['n']), mean0=map_state[0]['mean'],
            m2_0=map_state[0]['m2'],
            n1=np.asarray(map_state[1]['n']), mean1=map_state[1]['mean'],
            m2_1=map_state[1]['m2'],
            mapped_uids=np.asarray(
                sorted(mapped_uids), dtype=f'<U{uid_width}'
            ),
        )
    replace_with_retry(temporary_state, STATE_PATH)


completed = set(mapped_uids)
if not RETRY_FAILURES:
    completed |= {str(row['uid']) for row in failure_rows}
processed_since_checkpoint = 0

for row in tqdm(
    cohort.itertuples(index=False), total=len(cohort), desc='Full-cohort streaming'
):
    uid = str(row.uid)
    if uid in completed:
        continue
    try:
        if uid == REFERENCE_UID:
            registered_image = sitk.Image(reference_image)
            metrics = {
                'metric_final_correlation_objective': float('nan'),
                'correlation_before': 1.0, 'correlation_after': 1.0,
                'registration_backend': RESOLVED_REGISTRATION_BACKEND,
                'gpu_fallback_used': False,
                'registration_seconds': 0.0,
            }
        else:
            moving = read_sitk_from_zip(member_by_uid[uid])
            registered_image, _, metrics = register_to_reference(
                reference_image, moving, REGISTRATION_MODE
            )
        registered_array = sitk.GetArrayFromImage(registered_image).astype(np.float32)
        features, ratio, map_volume = feature_record(
            uid, registered_array, masks
        )
        features['is_pathologic'] = int(row.is_pathologic)
        features['acquisition_family'] = str(row.acquisition_family)
        feature_rows.append(features)
        failure_rows = [
            failure for failure in failure_rows
            if str(failure.get('uid')) != uid
        ]

        correlation_after = float(metrics['correlation_after'])
        correlation_before = float(metrics['correlation_before'])
        registration_rows.append({
            'uid': uid,
            'is_pathologic': int(row.is_pathologic),
            'acquisition_family': str(row.acquisition_family),
            **metrics,
            'correlation_gain': correlation_after - correlation_before,
            'registration_qc_flag': bool(
                (not np.isfinite(correlation_after))
                or (correlation_after < 0.55)
                or (correlation_after - correlation_before < -0.05)
            ),
        })
        online_update(int(row.is_pathologic), uid, map_volume)
        if SAVE_REGISTERED_CROPS:
            save_crop(uid, ratio, map_volume)
    except (KeyboardInterrupt, SystemExit):
        checkpoint()
        raise
    except Exception as error:
        failure_rows = [
            failure for failure in failure_rows
            if str(failure.get('uid')) != uid
        ]
        failure_rows.append({
            'uid': uid, 'error': f'{type(error).__name__}: {error}'
        })

    processed_since_checkpoint += 1
    if processed_since_checkpoint >= CHECKPOINT_EVERY:
        checkpoint()
        processed_since_checkpoint = 0

checkpoint()
features = pd.read_csv(FEATURES_PATH, dtype={'uid': 'string'})
registration_qc = pd.read_csv(REGISTRATION_PATH, dtype={'uid': 'string'})
failures = pd.read_csv(FAILURES_PATH, dtype={'uid': 'string'})
mean_registration_seconds = pd.to_numeric(
    registration_qc.get('registration_seconds'), errors='coerce'
).mean()
gpu_fallbacks = int(
    registration_qc.get(
        'gpu_fallback_used', pd.Series(False, index=registration_qc.index)
    ).fillna(False).astype(bool).sum()
)
background_valid = int(features['background_qc_valid'].astype(bool).sum())
background_floored = int(features['background_floor_applied'].astype(bool).sum())
display(Markdown(
    f'**Features:** `{len(features):,}/{len(cohort):,}` · '
    f'**fallos:** `{len(failures):,}` · '
    f'**crops:** `{len(list(CROPS_DIR.glob("*.npz"))) if SAVE_REGISTERED_CROPS else 0:,}` · '
    f'**backend:** `{RESOLVED_REGISTRATION_BACKEND}` · '
    f'**tiempo medio/registro:** `{mean_registration_seconds:.2f} s` · '
    f'**fallbacks GPU→CPU:** `{gpu_fallbacks}`'
))

## 7. Mapas online, efecto estandarizado y control visual

Las medias son voxel-a-voxel de volúmenes registrados y normalizados por el percentil 99 del
foreground; no son MIP ni dependen del denominador SBR.
Cohen d usa varianza agrupada ponderada por tamaño muestral. También se guarda Hedges g, que
corrige el sesgo de muestra pequeña.

In [ ]:
n0, n1 = map_state[0]['n'], map_state[1]['n']
normal_mean = map_state[0]['mean'].astype(np.float32)
pathologic_mean = map_state[1]['mean'].astype(np.float32)
normal_var = (
    map_state[0]['m2'] / max(n0 - 1, 1)
).astype(np.float32)
pathologic_var = (
    map_state[1]['m2'] / max(n1 - 1, 1)
).astype(np.float32)
difference = pathologic_mean - normal_mean
pooled_variance = (
    ((n0 - 1) * normal_var + (n1 - 1) * pathologic_var)
    / max(n0 + n1 - 2, 1)
)
pooled_std = np.sqrt(np.maximum(pooled_variance, 0))
cohen_d = np.divide(
    difference, pooled_std,
    out=np.zeros_like(difference), where=pooled_std > 1e-6,
)
correction = 1 - 3 / max(4 * (n0 + n1) - 9, 1)
hedges_g = correction * cohen_d

with MAPS_PATH.open('wb') as stream:
    np.savez_compressed(
        stream,
        normal_mean=normal_mean,
        pathologic_mean=pathologic_mean,
        difference=difference,
        pooled_std=pooled_std,
        cohen_d=cohen_d,
        hedges_g=hedges_g,
        n_normal=np.asarray(n0),
        n_pathologic=np.asarray(n1),
    )

target_z = int(np.argmax(masks['target'].sum(axis=(1, 2))))
map_z = int(np.argmax((np.abs(hedges_g) * masks['target']).sum(axis=(1, 2))))
figure, axes = plt.subplots(2, 4, figsize=(17, 8))
reference01 = normalize_display(consensus)
mask_overlay = np.dstack([
    reference01[target_z],
    masks['target'][target_z].astype(float),
    masks['background'][target_z].astype(float),
])
axes[0, 0].imshow(np.rot90(reference01[target_z]), cmap='hot', vmin=0, vmax=1)
axes[0, 0].set_title('Consenso')
axes[0, 1].imshow(np.rot90(mask_overlay), vmin=0, vmax=1)
axes[0, 1].set_title('Intensidad / target / fondo')
axes[0, 2].hist(registration_qc['correlation_after'].dropna(), bins=30)
axes[0, 2].set_title('Correlación posterior al registro')
sns.boxplot(
    data=features, x='is_pathologic',
    y='semiquant_log_target_background', ax=axes[0, 3]
)
axes[0, 3].set_title('log(1 + target/fondo robusto) por clase')
images = [normal_mean[map_z], pathologic_mean[map_z], difference[map_z], hedges_g[map_z]]
titles = ['Media normal', 'Media patológica', 'Diferencia', 'Hedges g descriptivo']
cmaps = ['hot', 'hot', 'coolwarm', 'coolwarm']
for axis, image, title, cmap in zip(axes[1], images, titles, cmaps):
    limit = np.nanpercentile(np.abs(image), 99) if cmap == 'coolwarm' else None
    axis.imshow(
        np.rot90(image), cmap=cmap,
        vmin=-limit if limit else None, vmax=limit if limit else None,
    )
    axis.set_title(title)
for axis in axes.ravel():
    if not axis.has_data() or axis not in [axes[0, 2], axes[0, 3]]:
        axis.axis('off')
plt.tight_layout()
plt.show()

coverage = pd.DataFrame({
    'indicador': [
        'Cohorte solicitada', 'Features extraídas', 'Fallos',
        'Normal en mapas', 'Patológica en mapas', 'Flags de registro',
        'Fallbacks GPU a CPU', 'Segundos medios por registro',
        'Fondos QC válidos', 'Piso de fondo aplicado',
    ],
    'valor': [
        len(cohort), len(features), len(failures), n0, n1,
        int(registration_qc['registration_qc_flag'].sum()),
        gpu_fallbacks, mean_registration_seconds,
        background_valid, background_floored,
    ],
})
display(coverage.style.hide(axis='index'))
display(Markdown('### Casos más sensibles a máscara/registro'))
stability_columns = [
    'uid', 'is_pathologic', 'stability_threshold_span',
    'stability_mask_span', 'stability_translation_span',
    'stability_background_span',
]
display(
    features.assign(
        stability_max=features[[
            'stability_threshold_span', 'stability_mask_span',
            'stability_translation_span', 'stability_background_span',
        ]].max(axis=1)
    ).nlargest(20, 'stability_max')[stability_columns + ['stability_max']]
    .style.hide(axis='index')
)
cleanup_registration_cache()

## 8. Contrato de salida

- `dat_radiomics_features_full.csv`: predictores candidatos por bloques.
- `registration_qc_full.csv`: QC y dominio; no es predictor biológico por defecto.
- `cohort_maps_full.npz`: medias, diferencia, Cohen d y Hedges g online.
- `registered_crops/*.npz`: crops normalizados para una futura rama de imagen.
- `registration_failures_full.csv`: denominador explícito y reintentos trazables.

El notebook no elimina casos ni declara biomarcadores clínicamente validados.